# 📈 Notebook 06: Estimasi & Regresi
## Analitika Data Keuangan Sektor Publik

**Tujuan Pembelajaran:**
1. Memahami konsep regresi linear sederhana dan berganda
2. Melatih dan mengevaluasi model regresi
3. Memvalidasi asumsi regresi
4. Menginterpretasikan koefisien model
5. Membuat prediksi nilai numerik (estimasi)

---
> **Problem:** Estimasi **nilai Realisasi APBD** berdasarkan Anggaran, PAD, dan faktor keuangan lainnya.

## 📦 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (11, 5)
sns.set_style('whitegrid')

print('Library berhasil dimuat ✅')

## 📂 2. Persiapan Data

In [ ]:
# Muat dan bersihkan data
df = pd.read_csv('../../Dataset/02-EDA/keuangan_pemda.csv')
df = df[(df['Anggaran_APBD'] > 0) & (df['Realisasi_APBD'] > 0)].copy()
df.dropna(subset=['Anggaran_APBD', 'Realisasi_APBD', 'PAD', 'Dana_Transfer',
                   'Belanja_Pegawai', 'Belanja_Modal'], inplace=True)

# Feature Engineering
df['Anggaran_M'] = df['Anggaran_APBD'] / 1e9      # Dalam Miliar
df['Realisasi_M'] = df['Realisasi_APBD'] / 1e9    # Dalam Miliar
df['PAD_M'] = df['PAD'] / 1e9
df['Transfer_M'] = df['Dana_Transfer'] / 1e9
df['BelPeg_M'] = df['Belanja_Pegawai'] / 1e9
df['BelMod_M'] = df['Belanja_Modal'] / 1e9

print(f'Data siap: {df.shape[0]} PEMDA')
df[['Kode_Pemda', 'Anggaran_M', 'Realisasi_M', 'PAD_M', 'Transfer_M']].head(5)

## 📐 3. Regresi Linear Sederhana (1 Variabel)

In [ ]:
# Definisi X dan y untuk regresi sederhana
X_simple = df[['Anggaran_M']]
y = df['Realisasi_M']

X_train, X_test, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

# Latih model
model_simple = LinearRegression()
model_simple.fit(X_train, y_train)
y_pred_simple = model_simple.predict(X_test)

print('REGRESI LINEAR SEDERHANA')
print('=' * 50)
print(f'Y = {model_simple.intercept_:.4f} + {model_simple.coef_[0]:.4f} × Anggaran')
print()
print(f'R²   : {r2_score(y_test, y_pred_simple):.4f}')
print(f'MAE  : Rp {mean_absolute_error(y_test, y_pred_simple):.2f} Miliar')
print(f'RMSE : Rp {np.sqrt(mean_squared_error(y_test, y_pred_simple)):.2f} Miliar')

In [ ]:
# Visualisasi regresi sederhana
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter + Regression Line
x_range = np.linspace(X_simple.min().values[0], X_simple.max().values[0], 100).reshape(-1, 1)
y_range = model_simple.predict(x_range)

axes[0].scatter(X_simple, y, alpha=0.5, color='#3498db', s=40, label='Data Aktual')
axes[0].plot(x_range, y_range, 'r-', linewidth=2.5, label='Garis Regresi')
axes[0].set_xlabel('Anggaran APBD (Miliar Rp)', fontsize=11)
axes[0].set_ylabel('Realisasi APBD (Miliar Rp)', fontsize=11)
axes[0].set_title(f'Regresi Sederhana\nY = {model_simple.intercept_:.2f} + {model_simple.coef_[0]:.3f}X', 
                  fontweight='bold')
axes[0].legend()

# Actual vs Predicted
axes[1].scatter(y_test, y_pred_simple, alpha=0.6, color='#2ecc71', s=50)
min_val = min(y_test.min(), y_pred_simple.min())
max_val = max(y_test.max(), y_pred_simple.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Garis Ideal')
axes[1].set_xlabel('Nilai Aktual (Miliar Rp)', fontsize=11)
axes[1].set_ylabel('Nilai Prediksi (Miliar Rp)', fontsize=11)
axes[1].set_title(f'Aktual vs Prediksi\nR² = {r2_score(y_test, y_pred_simple):.4f}', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 📊 4. Regresi Linear Berganda (Multiple Features)

In [ ]:
# Cek korelasi antar feature
feature_cols = ['Anggaran_M', 'PAD_M', 'Transfer_M', 'BelPeg_M', 'BelMod_M']

fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = df[feature_cols + ['Realisasi_M']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
            center=0, linewidths=0.5, square=True)
ax.set_title('Heatmap Korelasi Feature vs Target\n(Realisasi_M = Target)', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKorelasi dengan Realisasi_M (Target):')
print(corr_matrix['Realisasi_M'].sort_values(ascending=False))

In [ ]:
# Regresi Berganda
X_multi = df[feature_cols]
y_multi = df['Realisasi_M']

X_tr, X_te, y_tr, y_te = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

model_multi = LinearRegression()
model_multi.fit(X_tr, y_tr)
y_pred_multi = model_multi.predict(X_te)

print('REGRESI BERGANDA')
print('=' * 55)
print(f'Intercept (β₀): {model_multi.intercept_:.4f}')
print()
for feat, coef in zip(feature_cols, model_multi.coef_):
    print(f'  β({feat:12}): {coef:+.6f}  → Setiap +1 Miliar Rp, Realisasi berubah {coef:+.4f} Miliar')
print()
print(f'R²   : {r2_score(y_te, y_pred_multi):.4f} (model menjelaskan {r2_score(y_te, y_pred_multi)*100:.1f}% variasi)')
print(f'MAE  : Rp {mean_absolute_error(y_te, y_pred_multi):.3f} Miliar')
print(f'RMSE : Rp {np.sqrt(mean_squared_error(y_te, y_pred_multi)):.3f} Miliar')

## 🔎 5. Validasi Asumsi Regresi

In [ ]:
# Residual Analysis
residuals = y_te - y_pred_multi

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Residual vs Fitted
axes[0,0].scatter(y_pred_multi, residuals, alpha=0.6, color='#3498db', s=40)
axes[0,0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0,0].set_xlabel('Nilai Fitted (Prediksi)')
axes[0,0].set_ylabel('Residual')
axes[0,0].set_title('1. Residual vs Fitted\n(cek homoskedastisitas)', fontweight='bold')

# 2. Q-Q Plot (normalitas residual)
stats.probplot(residuals, plot=axes[0,1])
axes[0,1].set_title('2. Normal Q-Q Plot\n(cek normalitas residual)', fontweight='bold')

# 3. Histogram residual
axes[1,0].hist(residuals, bins=20, color='#9b59b6', edgecolor='white', linewidth=1.2)
axes[1,0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1,0].set_xlabel('Residual')
axes[1,0].set_ylabel('Frekuensi')
axes[1,0].set_title('3. Distribusi Residual\n(harusnya mendekati normal)', fontweight='bold')

# 4. Actual vs Predicted
axes[1,1].scatter(y_te, y_pred_multi, alpha=0.6, color='#2ecc71', s=40)
min_v, max_v = min(y_te.min(), y_pred_multi.min()), max(y_te.max(), y_pred_multi.max())
axes[1,1].plot([min_v, max_v], [min_v, max_v], 'r--', linewidth=2)
axes[1,1].set_xlabel('Aktual')
axes[1,1].set_ylabel('Prediksi')
axes[1,1].set_title(f'4. Aktual vs Prediksi\nR² = {r2_score(y_te, y_pred_multi):.4f}', fontweight='bold')

plt.suptitle('Diagnostic Plot — Validasi Asumsi Regresi', fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Cek Multikolinearitas dengan VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor

try:
    X_for_vif = X_tr.copy()
    vif_data = pd.DataFrame()
    vif_data['Feature'] = X_for_vif.columns
    vif_data['VIF'] = [variance_inflation_factor(X_for_vif.values, i)
                       for i in range(X_for_vif.shape[1])]
    vif_data = vif_data.sort_values('VIF', ascending=False)

    print('VIF (Variance Inflation Factor)')
    print('VIF > 10 → multikolinearitas bermasalah')
    print('-' * 35)
    for _, row in vif_data.iterrows():
        flag = '⚠️ TINGGI' if row['VIF'] > 10 else '✅ OK'
        print(f'  {row["Feature"]:15}: {row["VIF"]:6.2f}  {flag}')
except ImportError:
    print('statsmodels tidak terinstal. Jalankan: pip install statsmodels')
    print('Korelasi antar feature (sebagai indikasi multikolinearitas):')
    print(df[feature_cols].corr().round(3))

## 🔮 6. Estimasi APBD

In [ ]:
# Prediksi untuk PEMDA baru
# Feature: [Anggaran_M, PAD_M, Transfer_M, BelPeg_M, BelMod_M]
pemda_baru = pd.DataFrame({
    'Anggaran_M':   [8.5,  15.0,  3.2],
    'PAD_M':        [1.2,   3.5,  0.4],
    'Transfer_M':   [5.8,   8.0,  2.1],
    'BelPeg_M':     [2.8,   5.5,  1.2],
    'BelMod_M':     [1.5,   3.0,  0.6]
})

pred_realisasi = model_multi.predict(pemda_baru)

print('ESTIMASI REALISASI APBD UNTUK PEMDA BARU')
print('=' * 60)
for i, (pred, row) in enumerate(zip(pred_realisasi, pemda_baru.iterrows())):
    anggaran = row[1]['Anggaran_M']
    pct = pred / anggaran * 100
    print(f'PEMDA {i+1}:')
    print(f'  Anggaran        = Rp {anggaran:.1f} Miliar')
    print(f'  Estimasi Realisasi = Rp {pred:.2f} Miliar')
    print(f'  Est. % Realisasi   = {pct:.1f}%')
    print()

In [ ]:
# Perbandingan model: Linear, Ridge, Lasso
scaler_std = StandardScaler()
X_tr_std = scaler_std.fit_transform(X_tr)
X_te_std = scaler_std.transform(X_te)

models_compare = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1)
}

print('PERBANDINGAN MODEL REGRESI')
print('=' * 55)
print(f'{'Model':22} | {'R²':8} | {'MAE':12} | {'RMSE':12}')
print('-' * 55)

for name, m in models_compare.items():
    m.fit(X_tr_std, y_tr)
    preds = m.predict(X_te_std)
    r2 = r2_score(y_te, preds)
    mae = mean_absolute_error(y_te, preds)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    print(f'{name:22} | {r2:8.4f} | {mae:8.3f} M Rp | {rmse:8.3f} M Rp')

## 📝 7. Latihan

1. Tambahkan fitur `Tahun` ke model berganda. Apakah R² meningkat?
2. Coba **transformasi log** pada variabel yang skewnya tinggi. Bandingkan hasilnya.
3. Interpretasikan koefisien model: fitur mana yang paling berpengaruh terhadap realisasi?
4. Buat confidence interval untuk prediksi PEMDA baru menggunakan `statsmodels`.
5. **Diskusi:** Jika R² sangat tinggi (misalnya 0.98), apakah model pasti bagus? Apa risikinya?

In [ ]:
# ✏️ Ruang Eksplorasi — OLS dengan statsmodels
try:
    import statsmodels.api as sm
    X_sm = sm.add_constant(X_tr.values)
    model_sm = sm.OLS(y_tr.values, X_sm).fit()
    print(model_sm.summary())
except ImportError:
    print('statsmodels tidak terinstal. Jalankan: pip install statsmodels')